In [3]:
import {config} from "dotenv"
config({
    path: "../../.env"
})
console.log()
import {Client} from "@langchain/langgraph-sdk"

[dotenv@17.3.1] injecting env (0) from ../../.env -- tip: ⚙️  specify custom .env file path with { path: '/custom/path/.env' }



In [4]:
const lgclient = new Client({
    apiUrl: "http://localhost:2024"
})

In [5]:
import { type AgentState } from "../agents/agent.ts"
const ReadState = async (threadId: string) => {
    const state = await lgclient.threads.getState<AgentState>(threadId)
    console.log(state)
}

[dotenv@17.3.1] injecting env (0) from ../../.env -- tip: ⚙️  load multiple .env files with { path: ['.env.local', '.env'] }


In [10]:
const chat_old = async (threadId: string, message: string) => {
    const stream = lgclient.runs.stream(threadId, "agent", {
        input: { messages: [{ role: "user", content: message }] },
        streamMode: ["messages", "values"]
    })

    for await (const chunk of stream) {
        if (chunk.event === "messages/partial") {
            // Handle UI token streaming
            console.log(`[${chunk.event}]:${JSON.stringify(chunk.data,null,2)}`)
        } else if (chunk.event === "values") {
            // Handle strongly-typed AgentState logging
            const state = chunk.data as AgentState;
            console.log(`[${chunk.event}]:State\n${JSON.stringify(state,null,2)}`)
        }
    }
    // for await (const chunk of stream) {
    //     if(chunk.event === "messages/partial"){
    //         console.log(chunk.data[0]?.content)
    //     }
    // }
}

In [31]:
const chat = async (threadId: string, message: string) => {
    const stream = lgclient.runs.stream(threadId, "agent", {
        input: { messages: [{ role: "user", content: message }] },
        streamMode: ["messages", "values"]
    });

    // Print a starting prefix for the assistant
    process.stdout.write("🤖 Agent: ");

    for await (const chunk of stream) {
        if (chunk.event === "messages/partial") {
            // Extract the token content from the message chunk
            const token = chunk.data[0]?.content;
            
            // Print the token directly to standard output without a newline
            // console.log("[Token]: ", token?.toString())
            if (token) {
                console.clear()
                console.log(token);
            }
        } 
        // console.log(JSON.stringify(chunk, null, 2))
        // else if (chunk.event === "values") {
        //     const state = chunk.data as AgentState;
            
        //     // Add spacing so the state dump doesn't print on the same line as the streaming text
        //     console.log("\n\n[🔄 STATE UPDATE]:");
        //     console.log(JSON.stringify(state, null, 2));
            
        //     // Re-add the prefix if the agent continues typing after a state update (like a tool call)
        //     process.stdout.write("🤖 Agent: ");
        // }
    }
    
    // Add a final newline when the stream completely finishes
    console.log("\n");
}

In [7]:
export async function printThreadHistory(lgClient: Client, threadId: string): Promise<void> {
  try {
    // Fetch the state with our generic type for safety
    const state = await lgClient.threads.getState<AgentState>(threadId);
    const values = state.values;
    const messages = values.messages || [];

    console.log('\n==================================================');
    console.log(`🧵 THREAD ID: ${threadId}`);
    console.log('==================================================\n');

    if (messages.length === 0) {
      console.log('   (No messages in this thread yet)');
    }

    // 1. Print the Conversation Flow
    messages.forEach((msg) => {
      let header = `[${msg.type.toUpperCase()}]`;
      
      // Add emojis and specific formatting based on the type
      if (msg.type === 'tool') {
        header = `[🛠️ TOOL${msg.name ? `: ${msg.name}` : ''}]`;
      } else if (msg.type === 'user') {
        header = `[👤 USER]`;
      } else if (msg.type === 'ai' || msg.type === 'assistant') {
        header = `[🤖 ASSISTANT]`;
      } else if (msg.type === 'system') {
        header = `[⚙️ SYSTEM]`;
      }

      console.log(header);
      
      // Content can sometimes be an array (e.g., multimodal inputs or complex tool calls)
      if (typeof msg.content === 'string') {
        console.log(msg.content || '(empty message)');
      } else {
        console.log(JSON.stringify(msg.content, null, 2));
      }
      
      console.log('--------------------------------------------------');
    });

    console.log('\n==================================================');
    console.log('📦 FINAL STATE (Excluding Messages):');
    console.log('==================================================');
    
    // 2. Print the final state, omitting the messages array to keep the console clean
    const { messages: _omittedMessages, ...stateWithoutMessages } = values;
    
    if (Object.keys(stateWithoutMessages).length === 0) {
      console.log('   (No additional state variables)');
    } else {
      // console.dir is great for deep object inspection in Node/Bun
      console.dir(stateWithoutMessages, { depth: null, colors: true });
    }
    
    console.log('==================================================\n');

  } catch (error) {
    console.error(`\n❌ Error fetching thread ${threadId}:`);
    console.error(error instanceof Error ? error.message : error);
  }
}

In [8]:
export async function listServerResources(lgClient: Client): Promise<void> {
  try {
    console.log('\n==================================================');
    console.log('🌐 LANGGRAPH SERVER RESOURCES');
    console.log('==================================================\n');

    // ---------------------------------------------------------
    // 1. Fetch Agents (Assistants)
    // ---------------------------------------------------------
    console.log('🤖 AVAILABLE AGENTS (Assistants):');
    console.log('--------------------------------------------------');
    
    // search() without parameters fetches a paginated list of all assistants
    const assistants = await lgClient.assistants.search(); 

    if (assistants.length === 0) {
      console.log('   (No agents found on this server)');
    } else {
      assistants.forEach((assistant, index) => {
        console.log(`${index + 1}. Assistant ID: ${assistant.assistant_id}`);
        console.log(`   Graph ID:     ${assistant.graph_id}`); // The name of the compiled graph
        console.log(`   Created:      ${new Date(assistant.created_at).toLocaleString()}`);
        
        if (assistant.metadata && Object.keys(assistant.metadata).length > 0) {
          console.log(`   Metadata:`, assistant.metadata);
        }
        console.log(''); // Spacing
      });
    }

    console.log('==================================================\n');

    // ---------------------------------------------------------
    // 2. Fetch Threads
    // ---------------------------------------------------------
    console.log('🧵 ACTIVE THREADS:');
    console.log('--------------------------------------------------');
    
    // search() fetches recent threads. You can also pass metadata filters here.
    const threads = await lgClient.threads.search();

    if (threads.length === 0) {
      console.log('   (No threads found on this server)');
    } else {
      threads.forEach((thread, index) => {
        console.log(`${index + 1}. Thread ID: ${thread.thread_id}`);
        console.log(`   Status:    ${thread.status}`); // e.g., 'idle', 'busy'
        console.log(`   Created:   ${new Date(thread.created_at).toLocaleString()}`);
        
        if (thread.metadata && Object.keys(thread.metadata).length > 0) {
          console.log(`   Metadata:`, thread.metadata);
        }
        console.log(''); // Spacing
      });
    }

    console.log('==================================================\n');

  } catch (error) {
    console.error(`\n❌ Error fetching server resources:`);
    console.error(error instanceof Error ? error.message : error);
  }
}

In [9]:
await listServerResources(lgclient)


🌐 LANGGRAPH SERVER RESOURCES

🤖 AVAILABLE AGENTS (Assistants):
--------------------------------------------------
1. Assistant ID: fe096781-5601-53d2-b2f6-0d3403f7e9ca
   Graph ID:     agent
   Created:      4/1/2026, 3:44:53 AM
   Metadata: { created_by: "system" }


🧵 ACTIVE THREADS:
--------------------------------------------------
1. Thread ID: 019d49a7-a830-7770-905a-6983f5f7a0d1
   Status:    idle
   Created:   4/1/2026, 8:56:57 PM
   Metadata: {
  graph_id: "agent",
  assistant_id: "fe096781-5601-53d2-b2f6-0d3403f7e9ca"
}

2. Thread ID: 019d4988-b9d6-7112-bdcf-c1d6c9392c6e
   Status:    idle
   Created:   4/1/2026, 8:23:10 PM
   Metadata: {
  graph_id: "agent",
  assistant_id: "fe096781-5601-53d2-b2f6-0d3403f7e9ca"
}

3. Thread ID: 019d4980-b187-7112-bdcf-b27017496be6
   Status:    idle
   Created:   4/1/2026, 8:14:24 PM
   Metadata: {
  graph_id: "agent",
  assistant_id: "fe096781-5601-53d2-b2f6-0d3403f7e9ca"
}

4. Thread ID: 019d497f-799b-7112-bdcf-a1784eb24235
   Status:   

In [12]:
await printThreadHistory(lgclient, "019d49a7-a830-7770-905a-6983f5f7a0d1")


🧵 THREAD ID: 019d49a7-a830-7770-905a-6983f5f7a0d1

[HUMAN]
hi
--------------------------------------------------
[🤖 ASSISTANT]
Hi! How can I help you today?
--------------------------------------------------
[HUMAN]
what is the weather
--------------------------------------------------
[🤖 ASSISTANT]
I’d be happy to check the weather for you! Which city would you like the forecast for?
--------------------------------------------------
[HUMAN]
clear
--------------------------------------------------
[🤖 ASSISTANT]
Got it — no city specified, so I’ll skip the weather lookup.
--------------------------------------------------
[HUMAN]
Hi again whatsup?
--------------------------------------------------
[🤖 ASSISTANT]
Not much—just hanging out, ready to help. What’s on your mind?
--------------------------------------------------
[HUMAN]
Hi again whatsup?
--------------------------------------------------
[🤖 ASSISTANT]
Hey! Not much—just here and ready to help. What can I do for you today?
-

In [32]:
await chat("019d49a7-a830-7770-905a-6983f5f7a0d1","What is the weather in Himalay??")

I
I don
I don’t
I don’t have
I don’t have a
I don’t have a city
I don’t have a city named
I don’t have a city named “
I don’t have a city named “H
I don’t have a city named “Himalay
I don’t have a city named “Himalay”
I don’t have a city named “Himalay” in
I don’t have a city named “Himalay” in the
I don’t have a city named “Himalay” in the database
I don’t have a city named “Himalay” in the database.
I don’t have a city named “Himalay” in the database. Please
I don’t have a city named “Himalay” in the database. Please give
I don’t have a city named “Himalay” in the database. Please give me
I don’t have a city named “Himalay” in the database. Please give me a
I don’t have a city named “Himalay” in the database. Please give me a specific
I don’t have a city named “Himalay” in the database. Please give me a specific Himalayan
I don’t have a city named “Himalay” in the database. Please give me a specific Himalayan town
I don’t have a city named “Himalay” in the database. Please give me a 